Universidad del Valle de Guatemala  
Departamento de Ciencias de la Computación  
CC3084 - Data Science - sección 40

Integrantes:  
- Cristian Túnchez (231359)
- Dulce Ambrosio (231143)
- Daniel Chet (231177)
- Javier Linares (231135)

# 01. Carga y limpieza de los datos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se realiza el primer contacto con los datos de la competencia. El objetivo es responder tres preguntas antes de cualquier análisis: qué contiene el conjunto de datos, qué tan confiable es, y qué transformaciones necesita para poder ser analizado.

El trabajo se organiza de la siguiente manera:

1. **Carga y descripción de la estructura.** Cuántas observaciones y variables hay, y de qué tipo es cada una.
2. **Diagnóstico de calidad.** Valores faltantes, duplicados y consistencia interna.
3. **Limpieza y construcción de variables.** Las transformaciones necesarias y las variables nuevas que se derivan del texto.

El resultado se guarda en `data/processed/` y alimenta el cuaderno `02_eda.ipynb`.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto listo. La guía de configuración en Colab está en el [README](../README.md) principal.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

Token de Kaggle cargado desde los secretos de Colab.
Entorno: Google Colab
Raíz del proyecto: /content/math-misconception-map


## 1. Preparación del entorno

Se importan las bibliotecas necesarias y se añade la carpeta `src/` a la ruta de búsqueda de
Python para poder reutilizar las funciones del proyecto.

In [2]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import DATA_RAW
from src.data.make_dataset import asegurar_crudos, guardar
from src.visualization.visualize import set_estilo

set_estilo()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

print("Carpeta de datos crudos:", DATA_RAW)

Carpeta de datos crudos: /content/math-misconception-map/data/raw


## 2. Carga de los archivos

La competencia entrega tres archivos. El de entrenamiento es el único que trae las etiquetas y por tanto es el que se analiza. Los otros dos se cargan solo para entender el formato que se espera en la entrega.

In [3]:
# Descarga los archivos desde Kaggle si aún no están en disco.
asegurar_crudos()

train = pd.read_csv(DATA_RAW / "train.csv")
test = pd.read_csv(DATA_RAW / "test.csv")
sample_submission = pd.read_csv(DATA_RAW / "sample_submission.csv")

for nombre, df in [("train", train), ("test", test), ("sample_submission", sample_submission)]:
    print(f"{nombre:20s} filas: {df.shape[0]:>6,}   columnas: {df.shape[1]}")

Faltan ['train.csv', 'test.csv', 'sample_submission.csv'] en /content/math-misconception-map/data/raw.
Faltan ['train.csv', 'test.csv', 'sample_submission.csv']. Descargando desde Kaggle...
  train.csv: 7,936,617 bytes
  test.csv: 687 bytes
  sample_submission.csv: 240 bytes
train                filas: 36,696   columnas: 7
test                 filas:      3   columnas: 5
sample_submission    filas:      3   columnas: 2


In [4]:
print("Columnas de train:", list(train.columns))
print("Columnas de test: ", list(test.columns))
print()
print("Formato de entrega esperado:")
sample_submission

Columnas de train: ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation', 'Category', 'Misconception']
Columnas de test:  ['row_id', 'QuestionId', 'QuestionText', 'MC_Answer', 'StudentExplanation']

Formato de entrega esperado:


,row_id,Category:Misconception
0,36696,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
1,36697,True_Correct:NA False_Neither:NA False_Misconception:Incomplete
2,36698,True_Correct:NA False_Neither:NA False_Misconception:Incomplete


### ¿Qué se observa?

El conjunto de entrenamiento tiene **36,696 observaciones y 7 variables**. El archivo de prueba solo tiene 3 filas, porque es un marcador de posición (la competencia sustituye ese archivo por uno de aproximadamente 16,000 filas al momento de evaluar). Por eso todo el análisis se hace sobre `train`.

El archivo de prueba tiene 5 columnas en lugar de 7. Le faltan justamente `Category` y `Misconception`, que son las dos variables que hay que predecir.

El formato de entrega confirma que se espera una cadena `Category:Misconception` y se admiten **hasta tres predicciones separadas por espacio**. Es decir, la evaluación reconoce que la tarea es ambigua y premia tener la respuesta correcta entre las tres primeras candidatas.

Conviene notar que la documentación de la competencia no menciona la columna `row_id`, que sí está presente. Es simplemente el identificador de la fila.

## 3. Estructura de las variables

Se revisa el tipo de dato de cada variable, cuántos valores distintos toma y cuántos valores
faltantes tiene.

In [5]:
resumen = pd.DataFrame({
    "tipo_pandas": train.dtypes.astype(str),
    "valores_unicos": train.nunique(),
    "faltantes": train.isna().sum(),
    "pct_faltante": (train.isna().mean() * 100).round(2),
})
resumen

,tipo_pandas,valores_unicos,faltantes,pct_faltante
row_id,int64,36696,0,0.00
QuestionId,int64,15,0,0.00
QuestionText,object,15,0,0.00
MC_Answer,object,49,0,0.00
StudentExplanation,object,35726,0,0.00
Category,object,6,0,0.00
Misconception,object,35,26836,73.13


In [6]:
train.head(5)

,row_id,QuestionId,QuestionText,MC_Answer,StudentExplanation,Category,Misconception
0,0,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),0ne third is equal to tree nineth,True_Correct,NaN
1,1,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 / 3 because 6 over 9 is 2 thirds and 1 third is not shaded.,True_Correct,NaN
2,2,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),"1 3rd is half of 3 6th, so it is simplee to understand.",True_Neither,NaN
3,3,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 goes into everything and 3 goes into nine,True_Neither,NaN
4,4,31772,What fraction of the shape is not shaded? Give your answer in its simplest form. [Imag...,\( \frac{1}{3} \),1 out of every 3 isn't coloured,True_Correct,NaN


### Tipo y nivel de medición de cada variable

El tipo que asigna pandas no basta para saber cómo tratar una variable. Por ejemplo `QuestionId` es un número entero, pero sumarlo o promediarlo no tiene sentido porque es una etiqueta. Por eso se clasifica cada variable por su nivel de medición.

| Variable | Tipo en pandas | Nivel de medición | Descripción |
| --- | --- | --- | --- |
| `row_id` | entero | Identificador | Número de fila, único en todo el archivo |
| `QuestionId` | entero | Identificador | Identifica la pregunta diagnóstica |
| `QuestionText` | texto | Texto libre | Enunciado de la pregunta, transcrito desde una imagen |
| `MC_Answer` | texto | Nominal | Opción de respuesta que el estudiante seleccionó |
| `StudentExplanation` | texto | Texto libre | Justificación escrita por el estudiante |
| `Category` | texto | Nominal | Relación entre la respuesta elegida y la explicación |
| `Misconception` | texto | Nominal | Error conceptual específico, solo cuando aplica |

Solo hay dos variables de texto libre y ninguna variable numérica de razón.

La variable `Misconception` tiene **73.13% de valores faltantes**. Antes de decidir qué hacer con ellos hay que averiguar si son un problema de calidad o una característica del diseño.

## 4. Diagnóstico de calidad

### 4.1 Los valores faltantes de `Misconception`

La documentación indica que `Misconception` solo aplica cuando la categoría contiene un error conceptual. Se verifica si eso se cumple de forma exacta.

In [7]:
verificacion = pd.crosstab(
    train["Category"],
    train["Misconception"].notna().map({True: "tiene etiqueta", False: "vacío"}),
)
verificacion

Misconception,tiene etiqueta,vacío
Category,,
False_Correct,0,227
False_Misconception,9457,0
False_Neither,0,6542
True_Correct,0,14802
True_Misconception,403,0
True_Neither,0,5265


La correspondencia es perfecta. Las dos categorías que contienen la palabra `Misconception` tienen siempre una etiqueta, y las otras cuatro nunca la tienen.

**Decisión:** estos valores faltantes **no se imputan ni se eliminan**. No son datos perdidos, son un valor con significado propio: la ausencia de error conceptual. Se sustituyen por la categoría explícita `"NA"`, que es además la forma en que la competencia espera recibirlos en el archivo de entrega.

### 4.2 Registros duplicados

Se revisan tres niveles de duplicación, porque cada uno significa algo distinto.

In [8]:
print("Filas idénticas en todas las columnas:", train.duplicated().sum())
print("Explicaciones repetidas (texto idéntico):",
      train.duplicated(subset=["StudentExplanation"]).sum())

clave = ["QuestionId", "MC_Answer", "StudentExplanation"]
print("Combinaciones pregunta + respuesta + explicación repetidas:",
      train.duplicated(subset=clave).sum())

Filas idénticas en todas las columnas: 0
Explicaciones repetidas (texto idéntico): 970
Combinaciones pregunta + respuesta + explicación repetidas: 797


In [9]:
train["objetivo"] = train["Category"] + ":" + train["Misconception"].fillna("NA")

etiquetas_por_grupo = train.groupby(clave)["objetivo"].nunique()
print("Grupos idénticos en total:", len(etiquetas_por_grupo))
print("Grupos idénticos que recibieron más de una etiqueta:", (etiquetas_por_grupo > 1).sum())

ejemplo = etiquetas_por_grupo[etiquetas_por_grupo > 1].index[0]
train[
    (train["QuestionId"] == ejemplo[0])
    & (train["MC_Answer"] == ejemplo[1])
    & (train["StudentExplanation"] == ejemplo[2])
][["StudentExplanation", "Category", "Misconception"]]

Grupos idénticos en total: 35899
Grupos idénticos que recibieron más de una etiqueta: 61


,StudentExplanation,Category,Misconception
38,"1/3 of the 9 triangles are shaded, so 3/9=1/3.",True_Correct,NaN
39,"1/3 of the 9 triangles are shaded, so 3/9=1/3.",True_Neither,NaN


### ¿Qué se observa?

No hay ninguna fila idéntica en todas sus columnas, de modo que no hay duplicación técnica que corregir. Sí hay **970 explicaciones con texto repetido**, lo que es esperable, pues frases como "porque sí" o "lo calculé" las escriben muchos estudiantes distintos.

El hallazgo relevante está en el tercer nivel. Existen **61 grupos** en los que la misma pregunta, la misma opción elegida y exactamente la misma explicación recibieron etiquetas diferentes. Esto es ruido de anotación y una cota práctica: ningún modelo puede acertar en los dos casos de un grupo así, porque la entrada es idéntica.

**Decisión:** no se eliminan. Son observaciones legítimas y su cantidad, 61 sobre 36,696, es demasiado pequeña para afectar el análisis. Se conservan como evidencia del ruido de la tarea.

### 4.3 Consistencia entre el identificador y el texto de la pregunta

Se verifica que cada `QuestionId` corresponda siempre al mismo enunciado.

In [10]:
textos_por_pregunta = train.groupby("QuestionId")["QuestionText"].nunique()
print("Preguntas distintas:", train["QuestionId"].nunique())
print("Enunciados distintos por pregunta:", textos_por_pregunta.unique())

opciones_por_pregunta = train.groupby("QuestionId")["MC_Answer"].nunique()
print("Opciones de respuesta por pregunta:", opciones_por_pregunta.unique())
print("Opciones de respuesta distintas en total:", train["MC_Answer"].nunique())

Preguntas distintas: 15
Enunciados distintos por pregunta: [1]
Opciones de respuesta por pregunta: [4]
Opciones de respuesta distintas en total: 49


### ¿Qué se observa?

La correspondencia entre identificador y enunciado es uno a uno, así que no hay inconsistencias. Cada pregunta tiene exactamente cuatro opciones de respuesta, tal como describe el formato de pregunta diagnóstica.

**Solo hay 15 preguntas distintas** para casi 37,000 respuestas. El conjunto no es una muestra amplia de
problemas de matemática, es una muestra muy profunda de quince problemas. Esto condiciona todo lo que viene después y se analiza en detalle en el cuaderno de exploración.

También se observa que las opciones de respuesta distintas son 49 y no 60, que sería 15 preguntas por 4 opciones. Algunas opciones, típicamente números sencillos, se repiten entre preguntas distintas.

### 4.4 Consistencia de los nombres de las etiquetas

Las 35 etiquetas de error conceptual son cadenas de texto escritas por los anotadores. Se revisa si alguna aparece escrita de más de una forma.

In [11]:
etiquetas = train["Misconception"].dropna()
por_minusculas = etiquetas.str.lower().value_counts()
colisiones = {}
for nombre in etiquetas.unique():
    colisiones.setdefault(nombre.lower(), []).append(nombre)

print("Etiquetas distintas:", etiquetas.nunique())
print("Etiquetas distintas ignorando mayúsculas:", por_minusculas.size)
print()
for clave, variantes in colisiones.items():
    if len(variantes) > 1:
        print("Misma etiqueta escrita de varias formas:", variantes)

Etiquetas distintas: 35
Etiquetas distintas ignorando mayúsculas: 34

Misma etiqueta escrita de varias formas: ['Wrong_Fraction', 'Wrong_fraction']


In [12]:
duplicada = train[train["Misconception"].str.lower() == "wrong_fraction"]

print("En qué pregunta aparece cada variante:")
print(pd.crosstab(duplicada["Misconception"], duplicada["QuestionId"]).to_string())
print()
print("Ejemplos de cada variante:")
for variante in ["Wrong_fraction", "Wrong_Fraction"]:
    print(f"\n  {variante}:")
    for texto in duplicada[duplicada["Misconception"] == variante]["StudentExplanation"].head(2):
        print("   ", texto[:80])

En qué pregunta aparece cada variante:
QuestionId      31777  33471
Misconception               
Wrong_Fraction    273      0
Wrong_fraction      0    418

Ejemplos de cada variante:

  Wrong_fraction:
    24 divided by 8 equals 3 and 3 times 3 equals 9
    3/8 are worth 9. i took it away from 24

  Wrong_Fraction:
    I did 120 divided by 5 then that answer times 2
    1 / 5 equals 24 so 2 of those will be 48.


### ¿Qué se observa?

Aparece una inconsistencia que no es evidente a simple vista. La etiqueta **`Wrong_fraction`** y la etiqueta **`Wrong_Fraction`** difieren únicamente en una mayúscula, y describen el mismo error: usar una fracción distinta de la que pide el problema.

Lo revelador es cómo se reparten. `Wrong_Fraction` se usa **solo en la pregunta 31777** y `Wrong_fraction` **solo en la pregunta 33471**. Es decir, el equipo que anotó cada pregunta escribió el nombre de forma distinta, y el resultado es que **un mismo error conceptual figura en los datos como dos errores diferentes**.

Sin este hallazgo, las dos etiquetas contarían como ejemplos de errores atados a una única pregunta. Unificadas, pasan a ser un error que sí aparece en dos preguntas distintas, que es justo el tipo de evidencia que la competencia busca.

**Decisión:** la inconsistencia se **documenta pero no se corrige en los datos**. La razón es que la competencia evalúa contra las etiquetas tal como están escritas, de modo que reemplazarlas rompería la correspondencia con el objetivo oficial. La unificación es una decisión que corresponde a la fase de modelado, donde puede aplicarse de forma controlada y revertirse al momento de generar las predicciones.

## 5. Limpieza del texto

El texto de este conjunto viene de dos fuentes muy distintas y requiere tratamientos distintos.

### 5.1 Contenido matemático en notación LaTeX

Las opciones de respuesta y buena parte de los enunciados fueron transcritos desde imágenes y conservan la notación matemática en formato LaTeX.

In [13]:
print("Ejemplos de MC_Answer tal como vienen:")
for valor in train["MC_Answer"].drop_duplicates().head(6):
    print("  ", repr(valor))

print()
print("Proporción de MC_Answer con notación LaTeX:",
      round(train["MC_Answer"].str.contains(r"\\\(").mean(), 3))

preguntas = train.drop_duplicates("QuestionId")[["QuestionId", "QuestionText"]]
print("Enunciados con descripción de imagen:",
      preguntas["QuestionText"].str.contains(r"\[Image", regex=True).sum(), "de", len(preguntas))

Ejemplos de MC_Answer tal como vienen:
   '\\( \\frac{1}{3} \\)'
   '\\( \\frac{3}{6} \\)'
   '\\( \\frac{3}{8} \\)'
   '\\( \\frac{3}{9} \\)'
   '\\( 3 \\)'
   '\\( \\frac{1}{12} \\)'

Proporción de MC_Answer con notación LaTeX: 0.96
Enunciados con descripción de imagen: 3 de 15


In [14]:
def limpiar_latex(texto: str) -> str:
    '''Convierte la notación LaTeX de una respuesta en texto legible.

    Las fracciones pasan a la forma `a/b` y se retiran los delimitadores y las
    barras invertidas. El resultado sirve para leer y para contar caracteres,
    no para volver a representar la expresión.
    '''
    t = str(texto)
    t = re.sub(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", r"\1/\2", t)
    t = re.sub(r"\\[a-zA-Z]+", " ", t)
    t = t.replace("\\(", " ").replace("\\)", " ")
    t = re.sub(r"[{}$]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


train["MC_Answer_limpia"] = train["MC_Answer"].map(limpiar_latex)

train[["MC_Answer", "MC_Answer_limpia"]].drop_duplicates().head(8)

,MC_Answer,MC_Answer_limpia
0,\( \frac{1}{3} \),1/3
2348,\( \frac{3}{6} \),3/6
2760,\( \frac{3}{8} \),3/8
2878,\( \frac{3}{9} \),3/9
4857,\( 3 \),3
5516,\( \frac{1}{12} \),1/12
7685,\( \frac{6}{2} \),6/2
7972,\( 24 \),24


La limpieza deja las respuestas en una forma legible. La columna original **se conserva**, porque la notación LaTeX es exacta y la versión limpia es una simplificación.

### 5.2 Las explicaciones de los estudiantes

Las explicaciones son texto escrito por estudiantes de entre nueve y dieciséis años. Se revisa qué tipo de irregularidades contienen antes de decidir cómo tratarlas.

In [15]:
pd.set_option("display.max_colwidth", 120)
train["StudentExplanation"].sample(8, random_state=7).to_frame()

,StudentExplanation
20189,because the number with the most digest must be the biggest
9718,I got 1800 so I just chose the closest
5531,1/2 .don’t times the numerator only the denominator. 6x2=12=1/12
28563,144 divided by 10 = 84. It has 10 sides becausr the sum of interior angles = 1440.
1785,becausee it would be 3 / 6 and simplified as 1/3.
24471,i think this because 1/3 is equivalent to 5/15 and 2/5 is equivalent to 6/15 so if we add them up it is 11/15. a is...
14809,because in algebra when a number is next to a letter you need to times and 2×12=24 making the answer b.
8879,because one fifth is 20 and two fifths are 40 so thatt means it is 60.


### ¿Qué se observa y qué se decide?

El texto contiene errores de ortografía frecuentes ("becausr", "becausee", "thatt", "sidee", "digest" por "digits"), números escritos unas veces con cifras y otras con palabras, y uso irregular de mayúsculas y de signos de puntuación. Estas irregularidades **no se corrigen**, por dos razones. La primera es que corregirlas de forma automática introduciría errores nuevos sobre un texto que además mezcla lenguaje y notación matemática. La segunda, más importante, es que la forma de escribir puede ser en sí misma una señal: un estudiante que domina un concepto tiende a escribirlo de otra manera que uno que lo confunde.

La limpieza se limita entonces a lo que es seguro y reversible:

- Normalizar los espacios en blanco, colapsando espacios repetidos y saltos de línea.
- Eliminar espacios al inicio y al final.
- Crear una versión en minúsculas para los conteos de palabras, conservando el original.

In [16]:
train["explicacion_limpia"] = (
    train["StudentExplanation"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
)
train["explicacion_min"] = train["explicacion_limpia"].str.lower()

cambiadas = (train["explicacion_limpia"] != train["StudentExplanation"]).sum()
print(f"Explicaciones cuyo texto cambió al normalizar espacios: {cambiadas:,}")
print("Explicaciones que quedaron vacías tras limpiar:",
      (train["explicacion_limpia"].str.len() == 0).sum())

Explicaciones cuyo texto cambió al normalizar espacios: 2,410
Explicaciones que quedaron vacías tras limpiar: 0


## 6. Construcción de variables

Esta sección resuelve el problema señalado en la sección 3: no hay variables numéricas con las que trabajar, así que hay que derivarlas.

### 6.1 Descomposición de la variable objetivo

La variable `Category` combina dos decisiones distintas en una sola etiqueta. Separarlas permite estudiarlas por separado.

In [17]:
partes = train["Category"].str.split("_", n=1, expand=True)
train["respuesta_correcta"] = partes[0].map({"True": True, "False": False})
train["tipo_explicacion"] = partes[1]
train["misconception"] = train["Misconception"].fillna("NA")

print(train["Category"].value_counts().to_string())
print()
print("respuesta_correcta:", train["respuesta_correcta"].unique())
print("tipo_explicacion:  ", sorted(train["tipo_explicacion"].unique()))
print("objetivo completo, combinaciones distintas:", train["objetivo"].nunique())

Category
True_Correct           14802
False_Misconception     9457
False_Neither           6542
True_Neither            5265
True_Misconception       403
False_Correct            227

respuesta_correcta: [ True False]
tipo_explicacion:   ['Correct', 'Misconception', 'Neither']
objetivo completo, combinaciones distintas: 65


La variable `Category` se descompone en dos:

- `respuesta_correcta`: valor lógico que indica si la opción elegida por el estudiante era la correcta.
- `tipo_explicacion`: toma los valores `Correct`, `Misconception` o `Neither`, y describe qué revela la explicación.

La variable objetivo real de la competencia es la combinación de `Category` y `Misconception`, y toma **65 valores distintos**. Ese es el tamaño real del problema de clasificación.

### 6.2 Identificación de la respuesta correcta de cada pregunta

Como el prefijo de `Category` indica si la opción elegida era correcta, es posible reconstruir cuál es la opción correcta de cada pregunta.

In [18]:
clave_respuesta = (
    train[train["respuesta_correcta"]]
    .groupby("QuestionId")["MC_Answer"]
    .agg(lambda s: s.mode().iat[0])
    .rename("respuesta_correcta_pregunta")
)

comprobacion = (
    train[["QuestionId", "MC_Answer", "respuesta_correcta"]]
    .merge(clave_respuesta, on="QuestionId")
)
coincide = (
    comprobacion["MC_Answer"] == comprobacion["respuesta_correcta_pregunta"]
) == comprobacion["respuesta_correcta"]

print("Preguntas con respuesta correcta identificada:", clave_respuesta.notna().sum())
print("Filas donde la reconstrucción coincide con la etiqueta:",
      f"{coincide.mean() * 100:.2f}%")

Preguntas con respuesta correcta identificada: 15
Filas donde la reconstrucción coincide con la etiqueta: 99.95%


La reconstrucción es consistente en el **99.95%** de las filas. El 0.05% restante es observado, porque una inconsistencia en la clave de respuestas sería un problema serio.

In [19]:
train_rev = train.merge(clave_respuesta, on="QuestionId", how="left")
inconsistentes = train_rev[
    (train_rev["MC_Answer"] == train_rev["respuesta_correcta_pregunta"])
    != train_rev["respuesta_correcta"]
]

print("Filas inconsistentes:", len(inconsistentes))
print()
print(inconsistentes
      .groupby(["QuestionId", "MC_Answer", "respuesta_correcta"])
      .size().rename("filas").to_string())
print()
print("Opciones marcadas como correctas por pregunta:")
print(train[train["respuesta_correcta"]].groupby("QuestionId")["MC_Answer"].nunique()
      .value_counts().rename("preguntas").to_string())

Filas inconsistentes: 18

QuestionId  MC_Answer  respuesta_correcta
31778       \( 6 \)    False                  6
            \( 9 \)    True                  12

Opciones marcadas como correctas por pregunta:
MC_Answer
1    14
2     1


### ¿Qué se observa?

La inconsistencia se concentra en **una sola pregunta**, la 31778, y afecta a **18 filas**. En esa pregunta dos opciones distintas aparecen marcadas alguna vez como correctas, cuando por diseño solo una puede serlo. Las otras catorce preguntas tienen una única opción correcta sin ambigüedad.

Se trata por tanto de un error de anotación aislado, no de un problema sistemático. Es coherente con los 61 grupos de etiquetas contradictorias detectados en la sección 4.2 y refuerza la misma conclusión: **las etiquetas contienen ruido en una fracción pequeña pero real de los casos**.

**Decisión:** las 18 filas se conservan y se marcan mediante la variable `respuesta_correcta_pregunta`, que permite identificarlas en cualquier momento. Eliminarlas supondría decidir cuál de las dos anotaciones es la buena, algo que no puede establecerse sin volver a la pregunta original.

### 6.3 Variables derivadas del texto

Se construyen medidas cuantitativas sobre la explicación del estudiante. Cada una busca capturar un aspecto distinto de la forma de responder.

In [20]:
texto = train["explicacion_limpia"]

train["n_caracteres"] = texto.str.len()
train["n_palabras"] = texto.str.split().str.len()
train["long_media_palabra"] = (train["n_caracteres"] / train["n_palabras"]).round(2)
train["n_digitos"] = texto.str.count(r"\d")
train["tiene_digitos"] = train["n_digitos"] > 0
train["tiene_fraccion"] = texto.str.contains(r"\d\s*/\s*\d", regex=True)
train["tiene_simbolo_mat"] = texto.str.contains(r"[+\-*=<>%×÷]", regex=True)
train["tiene_interrogacion"] = texto.str.contains(r"\?", regex=True)
train["tiene_porque"] = train["explicacion_min"].str.contains(r"\b(?:because|cause|cos|coz|as)\b", regex=True)

derivadas = [
    "n_caracteres", "n_palabras", "long_media_palabra", "n_digitos",
    "tiene_digitos", "tiene_fraccion", "tiene_simbolo_mat",
    "tiene_interrogacion", "tiene_porque",
]
train[derivadas].head(8)

,n_caracteres,n_palabras,long_media_palabra,n_digitos,tiene_digitos,tiene_fraccion,tiene_simbolo_mat,tiene_interrogacion,tiene_porque
0,33,7,4.71,1,True,False,False,False,False
1,61,16,3.81,6,True,True,False,False,True
2,55,13,4.23,4,True,False,False,False,False
3,43,9,4.78,2,True,False,False,False,False
4,31,7,4.43,2,True,False,False,False,False
5,62,15,4.13,5,True,False,False,False,True
6,53,12,4.42,6,True,True,False,False,False
7,52,10,5.20,7,True,True,False,False,True


In [21]:
print("Variables numéricas derivadas:")
print(train[["n_caracteres", "n_palabras", "long_media_palabra", "n_digitos"]].describe().round(2).to_string())
print()
print("Variables lógicas derivadas (proporción de casos verdaderos):")
print(train[["tiene_digitos", "tiene_fraccion", "tiene_simbolo_mat",
             "tiene_interrogacion", "tiene_porque"]].mean().round(3).to_string())

Variables numéricas derivadas:
       n_caracteres  n_palabras  long_media_palabra  n_digitos
count      36696.00    36696.00            36696.00   36696.00
mean          69.78       15.36                4.56       4.72
std           38.44        8.12                0.67       4.61
min            1.00        1.00                1.00       0.00
25%           43.00       10.00                4.12       1.00
50%           60.00       13.00                4.54       4.00
75%           86.00       19.00                4.96       7.00
max          586.00      177.00               15.67     101.00

Variables lógicas derivadas (proporción de casos verdaderos):
tiene_digitos          0.783
tiene_fraccion         0.269
tiene_simbolo_mat      0.226
tiene_interrogacion    0.004
tiene_porque           0.546


### ¿Qué significa cada variable derivada?

| Variable | Qué captura |
| --- | --- |
| `n_caracteres`, `n_palabras` | Cuánto se esforzó el estudiante en justificar su respuesta |
| `long_media_palabra` | Distingue el texto redactado del que solo contiene cifras sueltas |
| `n_digitos`, `tiene_digitos` | Si la explicación se apoya en cálculo numérico |
| `tiene_fraccion` | Si aparece notación de fracción, central en varias de las preguntas |
| `tiene_simbolo_mat` | Si se usan operadores en lugar de describir la operación con palabras |
| `tiene_interrogacion` | Señal de duda o de respuesta evasiva |
| `tiene_porque` | Si la explicación llega a enunciar una causa, que es lo que se pide |

Los primeros resultados muestran que la explicación mediana tiene alrededor de 13 palabras, es decir una sola frase corta. Se trata de textos muy breves, lo que confirma la advertencia de la literatura sobre lo difícil que resulta clasificar texto corto con representaciones basadas en frecuencia de palabras.

## 7. Conjunto final y resguardo

Se ordenan las columnas, se descartan las auxiliares y se guarda el resultado.

In [22]:
columnas_finales = [
    "row_id", "QuestionId", "QuestionText",
    "MC_Answer", "MC_Answer_limpia",
    "StudentExplanation", "explicacion_limpia", "explicacion_min",
    "Category", "respuesta_correcta", "tipo_explicacion",
    "Misconception", "misconception", "objetivo",
    "n_caracteres", "n_palabras", "long_media_palabra", "n_digitos",
    "tiene_digitos", "tiene_fraccion", "tiene_simbolo_mat",
    "tiene_interrogacion", "tiene_porque",
]

train_limpio = train[columnas_finales].merge(clave_respuesta, on="QuestionId", how="left")

ruta = guardar(train_limpio, "train_limpio")
print("Archivo guardado en:", ruta)
print("Dimensiones finales:", train_limpio.shape)
print("Valores faltantes restantes:", int(train_limpio.drop(columns=["Misconception"]).isna().sum().sum()))

Archivo guardado en: /content/math-misconception-map/data/processed/train_limpio.parquet
Dimensiones finales: (36696, 24)
Valores faltantes restantes: 0


## 8. Resumen de la limpieza

| Aspecto revisado | Resultado | Decisión tomada |
| --- | --- | --- |
| Filas duplicadas exactas | Ninguna | No se requiere acción |
| Explicaciones con texto repetido | 970 | Se conservan, son respuestas frecuentes legítimas |
| Grupos idénticos con etiqueta distinta | 61 | Se conservan, documentan el ruido de anotación |
| Faltantes en `Misconception` | 73.13 % | No se imputan, se recodifican como `"NA"` por ser estructurales |
| Faltantes en las demás columnas | Ninguno | No se requiere acción |
| Clave de respuestas reconstruida | 18 filas inconsistentes en 1 pregunta | Se conservan y quedan identificables |
| Nombres de las etiquetas | `Wrong_fraction` y `Wrong_Fraction` son la misma | Se documenta, la unificación se deja al modelado |
| Notación LaTeX en las respuestas | 96 % de las filas | Se añade una versión legible, se conserva el original |
| Errores de ortografía en las explicaciones | Frecuentes | No se corrigen, pueden ser señal informativa |
| Espacios irregulares | 2,410 explicaciones afectadas | Se normalizan |
| Ausencia de variables numéricas | Confirmada | Se derivan nueve variables a partir del texto |

El conjunto pasa de 7 a 24 columnas y conserva las 36,696 observaciones. Todas las variables, originales y derivadas, quedan documentadas en `codebook.md`.

El análisis exploratorio continúa en `02_eda.ipynb`.